# TP7 - Hypothesis Testing

Estimated time: **1.5** hour


In this lab, you'll go over hands-on exercises using Python to further consolidate your understanding about hypothesis test, in particular how to formulate the null hypothesis and alternative hypothesis, how to interpret the result based on the p-value and significance level $a$. You are going to perform two-sided tests and one-sided test, using t-test, z-test and permutation test.

## Objectives


* Get familiar with the data set
* Sample exercises
  * Hypothesis tests: null hypothesis, alternative hypothesis, p-value, significance level
  * One-sided test and two-sided test
  * Hypothesis tests comparing means
  * Welch's t-test, permutation test
* Assignment questions

----


# Get familiar with the data set

### 📚 Import Libraries


Install required packages for this lab. `pip` will automatically skip already installed packages

In [ ]:
import importlib.util

# import name: pip installation name
required_pkgs = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
    "scipy": "scipy",
}

missing_pkgs = [
    pip_name
    for import_name, pip_name in required_pkgs.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_pkgs:
    print("⚠️ Missing packages:", ", ".join(missing_pkgs))
    packages = " ".join(missing_pkgs)
    %pip install $packages
else:
    print("🎉 All packages already installed.")

Import the libraries needed for the lab


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy.stats as stats

## 📊 Get Familiar with the Dataset

For this lab, we will make use of the *insurance* dataset stored in *insurance.csv*. It contains the medical cost claimed by the insured people to the health insurance company. For each insured people, it includes information such as age, sex, BMI, number of children, whether he/she smokes, the residence region.

| Column Name               | Description                                                                                                                                 |
|---------------------------|---------------------------------------------------------------------------------------------------------------------------------------------|
| **Age**                   | Age of the primary beneficiary of the health insurance.                                |
| **Sex**                   | Gender of the policy holder: male or female.                                                                                                                |
| **BMI**                   | Body Mass Index (BMI), which provides a sense of how over-weight or under-weight a person is relative to their height. <br>  BMI is calculated as the weight (in kilograms) divided by height (in meters) squared. An ideal BMI is within the range of 18.5 to 24.9.  |
| **Children**              | The number of children / dependents covered by the insurance plan.                                                                                  |
| **Smoker**                | Whether the insured person smokes regularly or not: Yes or No                              |
| **Region**                | The beneficiary's place of residence, divided into four geographic regions: northest, southest, southwest, or northwest                                                             |
| **Charges**               | Total medical costs billed to health insurance.                                                                                                |


### Load Dataset

In [ ]:
# Load the dataset
insurance = pd.read_csv('insurance.csv')  

### Display the dataset


In [ ]:
insurance

**Display summary statistics for columns with numeric data**

In [ ]:
insurance.describe()

**Show the histogram of columns with numeric data**

In [ ]:
cols = ["age", "bmi", "children", "charges"]

plt.figure(figsize=(20, 5))

for i, col in enumerate(cols, 1):
    plt.subplot(1, len(cols), i)
    sns.histplot(insurance[col], kde=False)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

**Show the histogram for categorical columns (sex, smoker, region)**

In [ ]:
categorical_cols = insurance.select_dtypes(include=['object']).columns

plt.figure(figsize=(12, 4))

for i, col in enumerate(categorical_cols, 1):
    plt.subplot(1, len(categorical_cols), i)
    sns.histplot(data=insurance, x=col, discrete=True)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()


# 📝 Example Exercises

# Two-sided hypothesis testing

## Example 1: Use hypothesis test to check whether there is statistically significant difference between the average BMI for male and female insured people, using a significance level of 5%.

#### Retrieve the BMI for females and males, and display summary statistics 

In [ ]:
# Retrieve BMI for female and male
bmi_female = insurance[insurance['sex']=='female']['bmi']
bmi_male = insurance[insurance['sex']=='male']['bmi']

# Get summary statistics for each group
bmi_female_stats = bmi_female.describe()
bmi_male_stats = bmi_male.describe()

# Create a new data frame to display statistics side-by-side
bmi_stats_df = pd.DataFrame({
    'Female': bmi_female_stats,
    'Male': bmi_male_stats
})

# Display summary statistics
bmi_stats_df

As shown above, the mean BMI for **females** is **30.38** and the mean BMI for **males** is **30.94**, giving an **observed sample difference of 0.56**.
Next, we will perform a hypothesis test to determine whether the observed difference is statistically significant and reflects a real difference in the true population mean BMI between males and females, or it is purely by random chance.

### <span style="color:blue"> Example 1-1: Define the null and alternative hypothesis <span>

Let $\mu_1, \mu_2$ represent the mean BMI of male insured people and female insured people, respectively

$H_0$: $\mu_1 = \mu_2$,  there is no difference between the mean BMI of male insured people and female insured people

$H_1$: $\mu_1 \neq \mu_2$,  there is difference between the mean BMI of male insured people and female insured people

---

❓<span style="color:purple;">**Quick test**</span>:
**Which statistical test is most suitable for checking whether the population mean BMI differs between male and female insured people? Select one answer.**

A. A one-sample t-test

B. A paired t-test

C. A two-sided Welch two-sample t-test

D. A one-sided z-test

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> <b>C. A two-sided Welch two-sample t-test.</b></p>

**Explanation**:

<ul style="color:blue;">
  <li><b>Outcome and comparison:</b> BMI is numeric, and we are comparing two unknown population means—mean BMI for males and mean BMI for females. A one-sample t-test is unsuitable because it compares one population mean with a fixed reference value.</li>
  <li><b>Study design:</b> Male and female insured people form two independent groups, so this is not a paired test.</li>
  <li><b>Direction:</b> The question asks whether the means differ in either direction, so the test is two-sided.</li>
  <li><b>Variance:</b> The population standard deviations are unknown, so a z-test is not suitable here. Welch's t-test is appropriate because it does not assume equal population variances.</li>
</ul>

</details>

---

### <span style="color:blue"> Example 1-2: Use Welch's t-test to perform two-sided test <span>

In [ ]:
import scipy.stats as stats

# Use Welch's t-test to perform the two-sided test
welch_result_1 = stats.ttest_ind(bmi_male,
                               bmi_female,
                               equal_var=False,
                               alternative="two-sided")

print("T-statistic:", welch_result_1.statistic)
print("Degrees of freedom:", welch_result_1.df)
print("P-value:", welch_result_1.pvalue)

<div style="background-color: #f4f8ff; border-left: 5px solid #3b82f6; border-radius: 6px; padding: 12px 16px; margin: 12px 0;">

🔧 What does the code do?

<ul>
  <li>In the above code, <code>ttest_ind()</code> from <code>scipy.stats</code> is used to perform a two-sided Welch two-sample t-test.</li>
  <li>The two samples are <code>bmi_male</code> and <code>bmi_female</code>.</li>
  <li>Setting <code>equal_var=False</code> selects Welch's test, which does not assume equal population variances.</li>
  <li>The returned result provides the <b>t-statistic</b>, <b>p-value</b>, and Welch <b>degrees of freedom</b> through the <code>.statistic</code>, <code>.pvalue</code>, and <code>.df</code> attributes.</li>
</ul>

</div>

How should we interpret the output?

| Output | Interpretation |
|---|---|
| **T-statistic** | The **T-statistic** measures the size of the difference between the means of two groups in units of standard error. <br>A positive t-statistic indicates that the mean of the first group is higher than the mean of the second group. |
| **P-value** | The **p-value** is the probability of observing a t-statistic at least as extreme as the one obtained, assuming the null hypothesis is true. |

---

❓<span style="color:purple;">**Quick test**</span>:
**According to the p-value in the output, assuming a significance level of 0.05, what conclusion should we draw: reject H0 or fail to reject H0?** 

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> Since P value is larger than the significance level 0.05 - hence, we <b>fail to reject the null hypothesis</b> at the 5% significance level.</p>

<p style="color:blue;"> In other words, there is no significant difference between the average BMI for male and female. The observed difference in the sample could be purely due to random chance. </p>

**Explanation**:

<p style="color:blue;"> A p-value above 0.05 indicates that the observed difference in sample means could reasonably occur under the assumption that the null hypothesis is true. Therefore, the data does not provide statistically significant evidence of a difference in the average BMI between males and females. The small difference observed in the sample may be due to <i>random sampling variation</i> rather than a real difference in the population. </p> 

</details>

---

### <span style="color:blue"> Example 1-3: Use permutation test from scipy.stats to perform two-sided test <span>

Permutation test is much more flexible. It is a **resampling-based, non-parametric** method that does not require a normality assumption. 

It assumes **independent observations** and, under $H_0$, treats the observations from the two groups as coming from one common distribution so that they can be randomly reassigned.

In [ ]:
from scipy.stats import permutation_test

# Define the function to compute the test statistic.
# Here, we are interested in difference in means.
def difference_in_means(x, y):
    return np.mean(x) - np.mean(y)

# Set a random seed for reproducibility
np.random.seed(3) 

# Perform permutation test using the one from scipy.stats
# The first parameter: the two samples considered for the permutation test
# Statistic: test statistic that will be calculated for each permutation
# n_resamples: number of random permutations (resamples) used to 
#              approximate the null distribution.
# alternative: the type of alternative hypothesis, could be "two sided",
#              for two-sided test, "greater" or "less " for one-sided test
perm_test_result_1 = permutation_test((bmi_male, bmi_female), 
                                      statistic=difference_in_means, 
                                      n_resamples=10000, 
                                      alternative='two-sided')

# Display the results
print("Observed difference in means:", perm_test_result_1.statistic)
print("Permutation Test p-value:", perm_test_result_1.pvalue)

In the above code, we make use of `permutation_test()` from the `scipy.stats` to perform permutation test. We need to provide the two groups of data: `bmi_male` and `bmi_female`. `n_resamples` indicate the number of resampling will be performed. We need to indicate the type of test: one-sided or two-sided, by setting the `alternative` parameter. The `statistic` is the function we defined to computes the statistic of interest.  

>Pay attention, unlike t-statistic in t-test, **the test statistic computed by permutation test is the characteristic of interest**, for instance here it is the difference in mean BMI between male and female.
>
>The **p-value** is calculated as the <span style="color:blue">number  of permuted test statistic, that are  as extreme as or more extreme than the observed test statistic, divided by the total number of permutations<span>.

As we can see in the output, **the p-value obtained with permutation test is similar to those obtained using Welch's t-test**. We can draw the same conclusion based on the p-values.

### <span style="color:blue"> Visualize the permutation distribution <span>

The permutation test function above returns:
- `.statistic`: the observed difference in sample means
  
- `.pvalue`: the estimated permutation-test p-value

- `.null_distribution`: the test statistics obtained for each reshuffling

Based on `.null_distribution` and `.statistic`, we can generate a figure similar to the one on lecture slide **#45**.

In [ ]:
null_dist_1 = perm_test_result_1.null_distribution  
observed_diff_1 = perm_test_result_1.statistic

# Visualization of the permuted differences
plt.figure(figsize=(8, 4))
# Plot a histogram of the permutation test statistics
plt.hist(null_dist_1, bins=30, alpha=0.7, color='skyblue', edgecolor='black', label='Permuted Statistics')
# Plot a vertical line corresponding to the observed difference 
plt.axvline(observed_diff_1, color='red', linestyle='dashed', linewidth=2, label='Observed Difference')
plt.title('Distribution of Permuted Differences in Means')
plt.xlabel('Difference in Means')
plt.ylabel('Frequency')
plt.legend()
plt.grid()
plt.show()

In the above plot, the **blue histogram** represents the **distribution of the permuted test statistics** (differences in means).

The **red dashed line** indicates the **observed test statistic**, namely the difference in the average BMI between male and female in the sample. As shown in the printed out, it is 0.5653794535118628.

# One-sided hypothesis test

## Example 2: use hypothesis test to check whether the average charges made by male is higher than female, using a significance level of 5%.

#### Retrieve charges claimed by male and female, display summary statistics

In [ ]:
# Retrieve charges claimed by male and female
female_charges = insurance[insurance['sex'] == "female"]['charges']
male_charges = insurance[insurance['sex'] == "male"]['charges']

# Get summary statistics for each group
female_charges_stats = female_charges.describe()
male_charges_stats = male_charges.describe()

# Create a new data frame to display statistics side-by-side
charges_stats_df = pd.DataFrame({
    'female charges': female_charges_stats,
    'male charges': male_charges_stats
})

# Display summary statistics
charges_stats_df

### <span style="color:blue"> Example 2-1:  Define the null and alternative hypothesis <span>

Let $\mu_1, \mu_2$ represent the average charges claimed by male insured people and female insured people, respectively

$H_0$: $\mu_1 \leq \mu_2 $,  the average charges claimed by male insured people is less than or equal to female insured people

$H_1$: $\mu_1 > \mu_2 $,  the average charges claimed by male insured people is larger than female insured people

### <span style="color:blue"> Example 2-2:  Use Welch's t-test to perform the one-sided test. <span>

We use again **Welch's two-sample t-test**, which does not assume that the two populations have equal variances, to perform the one-sided test. 

>This is preferable here because the sample standard deviations of charges differ between the two groups, as shown above.

The type of test to be conducted can be set via the **"alternative"** parameter, possible values include "**two-sided**" for two-sided test, "**greater**" or "**less**" for one-sided test.

In [ ]:
import scipy.stats as stats

# Perform a one-sided Welch two-sample t-test
# alternative='greater': specifies a one-sided test where the alternative hypothesis is mean of the 
# first sample (male_charges) is greater than the mean of the second sample (female_charges). 
welch_result_2 = stats.ttest_ind(male_charges,
                                 female_charges,
                                 equal_var=False,
                                 alternative="greater")

print("T-statistic:", welch_result_2.statistic)
print("Degrees of freedom:", welch_result_2.df)
print("One-sided p-value:", welch_result_2.pvalue)

In the above code, the `ttest_ind()` is used to perform the one-sided test. We need to provide the two groups of data, and also indicates which direction we are interested: greater or less, through the `alternative` parameter. 

>`alternative="greater"` specifies a one-sided test, where the alternative hypothesis is "mean of the **first sample** (male_charges) is greater than the mean of the **second sample** (female_charges)". 

-----

❓<span style="color:purple;">**Quick test**</span>:
**According to the p-value in the output, assuming a significance level of 0.05, what conclusion should we draw: reject H0 or fail to reject H0?**

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"> Since the p value is smaller than the significance level 0.05. Hence, we <b>reject the null hypothesis</b> at the 5% significance level..</p>

<p style="color:blue;"> In other words, there is sufficient statistical evidence to support the alternative hypothesis that, the average charges claimed by male is higher than female. </p>

**Explanation**:

<p style="color:blue;">Since the p-value is <b>smaller than 0.05</b>, the observed difference is very unlikely under the assumption that the null hypothesis is true. Therefore, the data provides <b>statistically significant evidence</b> against the null hypothesis. This indicates that the difference in average charges between males and females is <b>unlikely due to random sampling variation</b>, and instead supports the alternative hypothesis that males have higher average charges than females.</p>

</details>

-----

### <span style="color:blue"> Example 2-3:  Use permutation test from scipy.stats to perform the one-sided test <span>

Similarly, in permutation test, the type of test to be conducted can be set via the `alternative` parameter, possible values include "**two-sided**" for two-sided test, "**greater**" or "**less**" for one-sided test.

In [ ]:
from scipy.stats import permutation_test

# Define the function to compute the test statistic.
# Here, we are interested in difference in means.
def difference_in_means(x, y):
    return np.mean(x) - np.mean(y)

# Set a random seed for reproducibility
np.random.seed(3) 

# Perform permutation test using the one from scipy.stats
# The first parameter: the two samples considered for the permutation test
# Statistic: test statistic that will be calculated for each permutation
# n_resamples: number of random permutations (resamples) used to 
#              approximate the null distribution.
# alternative: the type of alternative hypothesis, could be "two sided",
#              for two-sided test, "greater" or "less " for one-sided test
perm_test_result_2 = permutation_test((male_charges, female_charges), 
                                      statistic=difference_in_means, 
                                      n_resamples=10000, 
                                      alternative='greater')

# Display the results
print("Observed difference in means:", perm_test_result_2.statistic)
print("Permutation Test p-value:", perm_test_result_2.pvalue)

Unlike t-statistic in Welch's t-test, **the test statistic computed by permutation test is the characteristic of interest**, for instance here it is the difference in mean charge claimed between male and female.

As we can see in the output, **the p-value obtained with permutation test is similar to those obtained using Welch's t-test**. We can draw the same conclusion based on the p-values.

### <span style="color:blue"> Visualize the permutation test result <span>

The permutation test done above also returns the test statistics (e.g. difference in mean charge claimed) computed in each resampling, which can be retrieved through the `.null_distribution` attribute in the return value. And the observed sample difference is stored in the `.statistic` attribute.

In [ ]:
null_dist_2 = perm_test_result_2.null_distribution 
observed_diff_2 = perm_test_result_2.statistic

# Visualization of the permuted differences
plt.figure(figsize=(8, 4))
# Plot a histogram of the permutation test statistics
plt.hist(null_dist_2, bins=30, alpha=0.7, color='skyblue', edgecolor='black', label='Permuted Statistics')
# Plot a vertical line corresponding to the observed difference 
plt.axvline(observed_diff_2, color='red', linestyle='dashed', linewidth=2, label='Observed Difference')
plt.title('Distribution of Permuted Differences in Means')
plt.xlabel('Difference in Means')
plt.ylabel('Frequency')
plt.legend()
plt.grid()
plt.show()

> **Note:** This is a **right-sided test** because the alternative hypothesis is
>
> $$
     H_1:\mu_1 > \mu_2.
  $$
>
> The red dashed line represents the observed difference, $T_{\text{obs}}$. Only permuted differences **at least as large as the observed difference** - that is, at or to the right of the red line - are counted when calculating the p-value:
>
> $$
    T^*\geq T_{\text{obs}}.
  $$

# Understanding permutation test

## Example 3: A step-by-step manual implementation of the one-sided permutation test

In this section, we develop our own code to perform the one-sided permutation test step-by-step. The implementation mirrors the algorithm and concept on lecture slides **#42-#46**. 

<p align="right">
  <img src="permutation_test.png" alt="permutation_test" width="400"/>
</p>

### Before we start: hypotheses and test statistic

Let $\mu_1$ and $\mu_2$ denote the population mean charges claimed by male insured people and female insured people, respectively.

- $H_0: \mu_1 \leq \mu_2$
- $H_1: \mu_1 > \mu_2$

Our **test statistic** is the difference in sample means, calculated in the order **male minus female** since we would like to test whether mean charge claimed by male is higher than female.

$$T = \bar{x}_{\text{male}} - \bar{x}_{\text{female}}$$

### <span style="color:blue"> Step 1 — Compute the observed test statistic

First, store the two samples and calculate the observed difference between their sample means. This value, $T_{\text{obs}}$, is the value that we will use for comparison with differences obtained in repeated reshuffling. 

In [ ]:
# Store the two original groups
group1 = male_charges
group2 = female_charges

# Step 1: Compute the observed test statistic (male mean - female mean)
observed_diff = np.mean(group1) - np.mean(group2)

print("Mean charges, male:", np.mean(group1))
print("Mean charges, female:", np.mean(group2))
print("Observed difference in means:", observed_diff)

### <span style="color:blue"> Step 2 — Combine all observations into one dataset

Next, we combine the male and female charge values into one dataset. In doing so, we treat all observations as if they were drawn from a single common distribution, as assumed under $H_0$. 

Then, we can randomly reshuffle the combined observations and divide them into two new groups while preserving the original group sizes. To this purpose, we also calculate the size of each group.

In [ ]:
# Step 2: Combine the observations into one dataset
combined = np.concatenate([group1, group2])

n_group1 = len(group1)
n_group2 = len(group2)

print("Group 1 size:", n_group1)
print("Group 2 size:", n_group2)
print("Combined sample size:", len(combined))

### <span style="color:blue"> Steps 3–5 — Work through one reshuffling

Before repeating the procedure thousands of times, we inspect one reshuffling:

1. Randomly shuffle the combined data.
2. Assign the first `n_group1` values to a new Group 1.
3. Assign the remaining values to a new Group 2.
4. Calculate and record the new difference in means, $T^*$.


In [ ]:
# Set a random seed for reproducibility
np.random.seed(3)

# Step 3: Randomly permute the combined data
permuted = np.random.permutation(combined)

# Step 4: Split the permuted data into two new groups
# preserving the original group size
new_group1 = permuted[:n_group1]
new_group2 = permuted[n_group1:]

# Step 5: Compute the difference in means for this reshuffling
one_permuted_diff = np.mean(new_group1) - np.mean(new_group2)

print("Difference in means for one reshuffling:", one_permuted_diff)

### <span style="color:blue"> Repeat the reshuffling many times

One reshuffling is not enough to describe random variation under $H_0$. We therefore repeat Steps 3–5 **10,000 times** and store every permuted difference, $T^*$. Together, these values approximate the **permutation distribution of the test statistic** under the null hypothesis.

In [ ]:
# Set the number of random reshufflings
num_permutations = 10000

# Reset the seed for a reproducible simulation
np.random.seed(6)

# Initialize a list to store the test statistic from each reshuffling
permuted_diffs = []

for _ in range(num_permutations):
    # Step 3: Randomly permute the combined data
    permuted = np.random.permutation(combined)

    # Step 4: Split the permuted data into two groups of the original sizes
    new_group1 = permuted[:n_group1]
    new_group2 = permuted[n_group1:]

    # Step 5: Calculate and store the difference in means
    permuted_diff = np.mean(new_group1) - np.mean(new_group2)
    permuted_diffs.append(permuted_diff)

print("Number of stored permuted differences:", len(permuted_diffs))

### <span style="color:blue"> Reading the p-value

<p align="right">
  <img src="p-value.png" alt="p-value" width="400"/>
</p>

For a (greater than) one-sided test $H_1: \mu_1 > \mu_2$, a permuted result is at least as extreme as the observed statistic when

$$T^* \geq T_{obs}.$$

We use `cnt` to record the number of reshufflings satisfying this condition and $B$ is the total number of reshufflings, the **estimated p-value** can be calculated as

$$p = \frac{cnt + 1}{B + 1}.$$

The extra 1 counts the original observed allocation alongside the random reshufflings. Because this is a one-sided test, only the **right tail** of the permutation distribution contributes to the p-value.

In [ ]:
# Count permuted differences at least as extreme as the observed difference
cnt = sum(
    1 for diff in permuted_diffs
    if diff >= observed_diff
)

p_value = (cnt + 1) / (num_permutations + 1)

print("Observed difference in means:", observed_diff)
print("Number of permutations:", num_permutations)
print("Number of statistics as large as or larger than observed:", cnt)
print("One-sided permutation-test p-value:", p_value)

### <span style="color:blue"> Make the statistical decision

Finally, we compare the one-sided p-value with the significance level $\alpha = 0.05$.

>- If $p \leq 0.05$, reject $H_0$.
>- If $p > 0.05$, fail to reject $H_0$.

**Answer**: The estimated p-value is approximately **0.017**, which is **smaller than the significance level of 5%**. Therefore, we **reject the null hypothesis $H_0$**. The data provide statistically significant evidence to support the alternative hypothesis - the population mean charges are higher for male insured people than for female insured people.

### <span style="color:blue"> Visualize the permutation distribution

In [ ]:
# Visualize the permutation distribution
plt.figure(figsize=(8, 4))  
plt.hist(
    permuted_diffs,               # Values to display in the histogram
    bins=30,                      # Number of intervals used to group the values
    alpha=0.7,                    # Transparency of the histogram bars
    color="skyblue",              # Fill color of the histogram bars
    edgecolor="black",            # Color of the bar outlines
    label="Permuted statistics"   # Label used for the histogram in the legend
)

# Mark the observed statistic and shade the relevant right tail
plt.axvline(
    observed_diff,                         # Horizontal position of the observed statistic
    color="red",                           # Color of the vertical reference line
    linestyle="dashed",                    # Display the reference line as dashed
    linewidth=2,                           # Thickness of the reference line
    label=r"Observed difference $T_{obs}$" # Label used for the line in the legend
)
plt.axvspan(
    observed_diff,                              # Left boundary of the shaded right tail
    max(permuted_diffs),                        # Right boundary of the shaded right tail
    color="red",                                # Color of the shaded region
    alpha=0.15,                                 # Transparency of the shaded region
    label="Statistics contributing to the p-value" # Label used for the region in the legend
)

plt.title("Permutation Distribution of Differences in Mean Charges")   
plt.xlabel("Permuted difference in means (male - female)")             
plt.ylabel("Frequency")                                                
plt.legend()                                                           
plt.grid(alpha=0.3)                                                    
plt.show()                                                             

The histogram above approximates the null distribution of the difference in mean charges between male and female insured people. 

The **red dashed line** marks the observed test statistic, $T_{obs}$. 

Since this is a one-sided test, more specifically the alternative hypothesis is right-sided (or greater than), only permuted statistics as large as or larger than the observed statistic $T_{obs}$ contribute to the p-value. These statistics correspond to the **red shaded region** in the plot .

---

❓<span style="color:purple;">**Quick test:**</span> Suppose we changed the research question from **whether mean charges are higher for males** to **whether mean charges differ between males and females**. Which part of the manual permutation test would need to change?

A. How the two samples are combined

B. How the observations are randomly reshuffled

C. The sizes of the two new groups

D. The definition of an extreme result and the boundaries shown on the graph

<details> <summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;"><b>D. The definition of an extreme result and the boundaries shown on the graph.</b></p>

**Explanation**:

<p style="color:blue;">The reshuffling algorithm remains unchanged: we combine the observations, randomly reshuffle them, and form two new groups with the original group sizes. Only the definition of an extreme result changes.</p>

<ul style="color:blue;">
  <li><b>One-sided:</b> $T^* \geq T_{obs}$</li>
  <li><b>Two-sided:</b> $|T^*| \geq |T_{obs}|$</li>
</ul>

<p style="color:blue;">For a two-sided test, differences in either direction can provide evidence against $H_0$. The graph would therefore mark both $-|T_{obs}|$ and $+|T_{obs}|$, and permuted statistics beyond either boundary would contribute to the p-value.</p>

</details>

---

# Optional material <span style="color:purple;"> [for advanced students only] </span>

## Manual implementation of Welch's two-sample t-test

The Welch t-statistic compares the observed difference in sample means with the standard error of that difference:

$$
t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$

The following function implements this formula, calculates the Welch–Satterthwaite degrees of freedom, and obtains the p-value from the t-distribution.

In [ ]:
def welch_ttest_manual(sample1, sample2, alternative="two-sided"):
    # Convert both samples to NumPy arrays
    sample1 = np.asarray(sample1, dtype=float)
    sample2 = np.asarray(sample2, dtype=float)

    # Calculate the sample sizes, means, and unbiased sample variances
    n1, n2 = len(sample1), len(sample2)
    mean1, mean2 = np.mean(sample1), np.mean(sample2)
    variance1, variance2 = np.var(sample1, ddof=1), np.var(sample2, ddof=1)

    # Calculate the observed difference and its standard error
    observed_difference = mean1 - mean2
    standard_error = np.sqrt(variance1 / n1 + variance2 / n2)

    # Calculate the Welch t-statistic
    t_statistic = observed_difference / standard_error

    # Calculate the Welch–Satterthwaite degrees of freedom
    numerator = (variance1 / n1 + variance2 / n2) ** 2
    denominator = (variance1 / n1) ** 2 / (n1 - 1) + (variance2 / n2) ** 2 / (n2 - 1)
    degrees_of_freedom = numerator / denominator

    # Calculate the p-value from the t-distribution
    if alternative == "two-sided":
        p_value = 2 * stats.t.sf(abs(t_statistic), degrees_of_freedom)
    elif alternative == "greater":
        p_value = stats.t.sf(t_statistic, degrees_of_freedom)
    elif alternative == "less":
        p_value = stats.t.cdf(t_statistic, degrees_of_freedom)
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")

    return observed_difference, standard_error, t_statistic, degrees_of_freedom, p_value

<div style="background-color: #f4f8ff; border-left: 5px solid #3b82f6; border-radius: 6px; padding: 12px 16px; margin: 12px 0;">

<b>🔍 How does the manual function work?</b>

<ol>
  <li><b>Prepare the samples:</b> Convert both samples to NumPy arrays, then calculate their sizes, means, and unbiased sample variances. Setting <code>ddof=1</code> gives the sample variance.</li>
  <li><b>Calculate the observed difference:</b> Subtract the mean of the second sample from the mean of the first sample.</li>
  <li><b>Calculate the standard error:</b> Combine the two variance contributions using <code>sqrt(variance1 / n1 + variance2 / n2)</code>.</li>
  <li><b>Calculate the t-statistic:</b> Divide the observed difference by its standard error.</li>
  <li><b>Complete the test:</b> Calculate the Welch–Satterthwaite degrees of freedom and use the t-distribution to obtain the p-value for the selected alternative.</li>
</ol>

<p><b>Returned values:</b> observed difference, standard error, t-statistic, degrees of freedom, and p-value.</p>

</div>

In [ ]:
# Call the manual implementation for the male and female BMI samples
mean_difference, standard_error, t_statistic, df, p_value = welch_ttest_manual(
    bmi_male,
    bmi_female,
    alternative="two-sided"
)

#print("Observed difference in means:", mean_difference)
#print("Standard error:", standard_error)
print("\nT-statistic:", t_statistic)
print("Degrees of freedom:", df)
print("P-value:", p_value)

As we can see, the manual implementation obtains very close result as the Welch's t-test.

# 🏠 Assignment Questions <span style="color:blue;"> [20/20 points] </span>

## <span style="color:blue;"> [6 points] </span> Question 1: use hypothesis test to check whether the average BMI of insured people without children is different than the insured people with 1 or 2 children, using a significance level of 5%.

### Q1-1: Retrieve the BMI for insured people without any children and with 1 or 2 children, and display the summary statistics 

In [ ]:
# insert your code here

###  Q1-2: Define the null and alternative hypothesis 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q1-3:  Use Welch's t-test to perform two-sided test, and print out the t-statistic and the p-value 

In [ ]:
# insert your code here

### Q1-4: based on the p-value, should we reject the null hypothesis or not? Explain your answer. 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q1-5: Use permutation test from scipy.stats to perform two-sided test, and print out the observed difference in means and the p-value

In [ ]:
# insert your code here

#### Question: answer does it provide consistent result as the test done using the Welch's t-test? 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q1-6: Visualize the permutation distribution

In [ ]:
# insert your code here

## <span style="color:blue;"> [8 points] </span> Question 2: implement your manual permutation test to check whether the average BMI of insured people without children is different than the insured people with 1 or 2 children, using a significance level of 5%. 

### Q2-1: Compute and print the observed test statistic

In [ ]:
# insert your code here

### Q2-2: Combine all observations into one dataset

In [ ]:
# insert your code here

### Q2-3: Perform the reshuffling, repeat 10,000 times and store the test statistic from each reshuffling

In [ ]:
# insert your code here

### Q2-4: Calculate the p-value for the two-sided test.
**Hint**: you can refer to the formula in the Example exercise to determine how to calculate the p-value for two-sided test.

In [ ]:
# insert your code here

### Q2-5: Make the statistical decision: reject $H_0$ or fail to reject $H_0$. Justify your answer.

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q2-6: Visualize the permutation distribution

**Requirement**: since this is two-sided test, in addition to the histogram and the observed test statistic, you also need to **highlight the two regions** contributing to the computation of the p-value.

In [ ]:
# insert your code here

## <span style="color:blue;"> [6 points] </span> Question 3: use hypothesis test to check whether the average BMI of insured younger people (age with the range [20, 30]) is lower than insured older people (age with the range [40, 50]), using a significance level of 5%.

### Q3-1: Retrieve BMI for younger people and older people, display the summary statistic. 

In [ ]:
# insert your code here

### Q3-2: Define the null and alternative hypothesis 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q3-3: Use Welch's t-test to perform the one-sided test, and print out the t-statistic and the p-value.  

In [ ]:
# insert your code here

### Q3-4: based on the p-value, should we reject the null hypothesis or not? Explain your answer. 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q3-5: Use permutation test from scipy.stats to perform the one-sided test, and print out the observed difference in means and the p-value

In [ ]:
# insert your code here

#### Question: answer does it provide consistent result as the test done using the Welch's t-test? 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### Q3-6: Visualize the permutation distribution

In [ ]:
# insert your code here